# TELECOM CAPSTONE PROJECT
## IBM Telco Intelligent Agent
### Beginner-Friendly Step-by-Step Hands-On Lab using LangGraph, LangSmith Concepts, Milvus RAG, and MCP Thinking

This notebook is a beginner-friendly capstone project lab for building a telecom intelligent agent.

---

## Project Goal

Build a small **IBM Telco Intelligent Agent** that handles a request like:

> "My mobile internet is slow in Pune and my bill is high. What should support do next?"

The flow is:

```text
User Query
   ↓
Planner
   ↓
Tools
   - KPI API
   - Incident API
   - Billing API
   ↓
RAG (SOP)
   ↓
Decision Node
   ↓
Response / Escalation
```

---

## What you will learn

### Design
- use case definition
- workflow thinking
- MCP-style tool design

### Build
- LangGraph workflow
- telecom tools
- Milvus-style RAG layer
- decision logic

### Observability
- LangSmith concepts
- tracing structure
- cost and performance logging

### Evaluation
- accuracy
- cost
- performance

## Step 1 — Use case definition

A telecom support assistant should:
1. understand the customer problem
2. look at customer KPI data
3. check current incidents
4. check billing signals
5. retrieve SOP guidance
6. decide whether to:
   - respond normally
   - recommend self-help
   - escalate to NOC
   - escalate to billing support

In [ ]:
# Uncomment if needed in a fresh environment
# %pip install -U langgraph pandas

## Step 2 — Imports

In [1]:
from __future__ import annotations

from typing import TypedDict, Dict, Any, List, Optional
from pprint import pprint
import pandas as pd

from langgraph.graph import StateGraph, END

## Step 3 — Beginner concepts

### LangGraph
LangGraph helps build workflows using:
- state
- nodes
- edges

### Tools
Tools are external functions the workflow can call.

### RAG
RAG means:
- retrieve relevant knowledge
- use it to improve decision quality

### MCP-style thinking
MCP helps us think in terms of:
- tools
- resources
- prompts

### LangSmith concepts
LangSmith helps with:
- tracing
- observability
- debugging
- cost/performance review

## Step 4 — MCP-style project design

In [2]:
mcp_design = {
    "tools": [
        {"name": "kpi_api", "description": "Returns telecom KPI signals such as latency, packet loss, and tower utilization"},
        {"name": "incident_api", "description": "Returns outage or live incident data"},
        {"name": "billing_api", "description": "Returns billing anomaly and plan usage data"},
    ],
    "resources": [
        {"name": "sop_knowledge_base", "description": "Telecom SOP and RCA guidance store"},
    ],
    "prompts": [
        {"name": "telecom_diagnosis_prompt", "description": "Prompt template for telecom diagnosis and escalation advice"},
    ],
}

pprint(mcp_design)

{'prompts': [{'description': 'Prompt template for telecom diagnosis and '
                             'escalation advice',
              'name': 'telecom_diagnosis_prompt'}],
 'resources': [{'description': 'Telecom SOP and RCA guidance store',
                'name': 'sop_knowledge_base'}],
 'tools': [{'description': 'Returns telecom KPI signals such as latency, '
                           'packet loss, and tower utilization',
            'name': 'kpi_api'},
           {'description': 'Returns outage or live incident data',
            'name': 'incident_api'},
           {'description': 'Returns billing anomaly and plan usage data',
            'name': 'billing_api'}]}


## Step 5 — Mock telecom enterprise data

In [3]:
CUSTOMERS = {
    "CUST_1001": {
        "customer_id": "CUST_1001",
        "name": "Ravi Patil",
        "city": "Pune",
        "plan": "Premium 5G",
        "recent_complaints": 2,
    },
    "CUST_2001": {
        "customer_id": "CUST_2001",
        "name": "Meera Shah",
        "city": "Mumbai",
        "plan": "Standard 4G",
        "recent_complaints": 0,
    },
}

KPI_DATA = {
    ("Pune", "slow_internet"): {
        "latency_ms": 210,
        "packet_loss_pct": 6,
        "tower_utilization_pct": 93,
        "signal_quality": "moderate",
    },
    ("Mumbai", "packet_loss"): {
        "latency_ms": 180,
        "packet_loss_pct": 9,
        "tower_utilization_pct": 72,
        "signal_quality": "good",
    },
}

INCIDENT_DATA = {
    ("Pune", "slow_internet"): {
        "incident_active": False,
        "incident_summary": "No citywide outage found. Local congestion patterns observed during peak hours."
    },
    ("Mumbai", "packet_loss"): {
        "incident_active": True,
        "incident_summary": "Packet loss incident active in Mumbai region."
    },
}

BILLING_DATA = {
    "CUST_1001": {
        "billing_anomaly": True,
        "usage_spike": True,
        "billing_summary": "Data overage charge detected after high weekend usage."
    },
    "CUST_2001": {
        "billing_anomaly": False,
        "usage_spike": False,
        "billing_summary": "Billing normal."
    },
}

SOP_DOCS = [
    {
        "doc_id": "SOP001",
        "city": "Pune",
        "issue_type": "slow_internet",
        "content": "If tower utilization exceeds 90 percent during peak hours, classify the issue as likely congestion and notify NOC."
    },
    {
        "doc_id": "SOP002",
        "city": "Pune",
        "issue_type": "slow_internet",
        "content": "If repeated complaints continue for premium customers, escalate after validating KPI and incident data."
    },
    {
        "doc_id": "SOP003",
        "city": "Mumbai",
        "issue_type": "packet_loss",
        "content": "If packet loss is confirmed and incident is active, communicate outage status and escalate to network operations."
    },
    {
        "doc_id": "SOP004",
        "city": "Generic",
        "issue_type": "billing_issue",
        "content": "If unusual bill is reported, verify usage spike, overage, active plan, and raise billing support case if needed."
    },
]

pd.DataFrame(SOP_DOCS)

,doc_id,city,issue_type,content
0,SOP001,Pune,slow_internet,If tower utilization exceeds 90 percent during...
1,SOP002,Pune,slow_internet,If repeated complaints continue for premium cu...
2,SOP003,Mumbai,packet_loss,If packet loss is confirmed and incident is ac...
3,SOP004,Generic,billing_issue,"If unusual bill is reported, verify usage spik..."


## Step 6 — Mock Milvus / RAG retrieval

In [4]:
def mock_rag_search(query_text: str, city_filter: Optional[str] = None, issue_filter: Optional[str] = None, top_k: int = 2):
    query_terms = set(query_text.lower().split())
    scored = []

    for doc in SOP_DOCS:
        if city_filter and doc["city"] not in [city_filter, "Generic"]:
            continue
        if issue_filter and doc["issue_type"] not in [issue_filter, "billing_issue"]:
            continue

        score = len(query_terms & set(doc["content"].lower().split()))
        scored.append({**doc, "score": score})

    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]

mock_rag_search("slow internet in Pune with high bill", city_filter="Pune", issue_filter="slow_internet")

[{'doc_id': 'SOP004',
  'city': 'Generic',
  'issue_type': 'billing_issue',
  'content': 'If unusual bill is reported, verify usage spike, overage, active plan, and raise billing support case if needed.',
  'score': 1},
 {'doc_id': 'SOP001',
  'city': 'Pune',
  'issue_type': 'slow_internet',
  'content': 'If tower utilization exceeds 90 percent during peak hours, classify the issue as likely congestion and notify NOC.',
  'score': 0}]

## Step 7 — Tool functions

In [5]:
def kpi_api(city: str, issue_type: str) -> Dict[str, Any]:
    return KPI_DATA.get(
        (city, issue_type),
        {
            "latency_ms": 120,
            "packet_loss_pct": 1,
            "tower_utilization_pct": 55,
            "signal_quality": "good",
        },
    )

def incident_api(city: str, issue_type: str) -> Dict[str, Any]:
    return INCIDENT_DATA.get(
        (city, issue_type),
        {
            "incident_active": False,
            "incident_summary": "No active incident found.",
        },
    )

def billing_api(customer_id: str) -> Dict[str, Any]:
    return BILLING_DATA.get(
        customer_id,
        {
            "billing_anomaly": False,
            "usage_spike": False,
            "billing_summary": "Billing normal.",
        },
    )

## Step 8 — LangSmith-style tracing concepts

In [6]:
TRACE_STORE = []

def start_trace(trace_name: str, initial_input: Dict[str, Any]) -> Dict[str, Any]:
    trace = {
        "trace_id": f"trace_{len(TRACE_STORE) + 1}",
        "trace_name": trace_name,
        "initial_input": initial_input,
        "runs": [],
        "total_cost_usd": 0.0,
        "total_prompt_tokens": 0,
        "total_completion_tokens": 0,
    }
    TRACE_STORE.append(trace)
    return trace

def add_run(trace: Dict[str, Any], node_name: str, node_input: Dict[str, Any], node_output: Dict[str, Any], prompt_tokens: int = 0, completion_tokens: int = 0):
    run = {
        "node_name": node_name,
        "input": node_input,
        "output": node_output,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
    }
    trace["runs"].append(run)
    trace["total_prompt_tokens"] += prompt_tokens
    trace["total_completion_tokens"] += completion_tokens
    trace["total_cost_usd"] += round((prompt_tokens / 1000) * 0.0005 + (completion_tokens / 1000) * 0.0015, 6)

## Step 9 — Cost and performance helpers

In [7]:
def estimate_tokens(text: str) -> int:
    return max(1, len(text.split()))

def estimate_latency_ms(node_name: str) -> int:
    latency_map = {
        "planner": 15,
        "kpi_tool": 35,
        "incident_tool": 30,
        "billing_tool": 25,
        "rag_retrieval": 40,
        "decision": 55,
        "response": 45,
        "evaluation": 20,
    }
    return latency_map.get(node_name, 25)

## Step 10 — LangGraph state

In [8]:
class CapstoneState(TypedDict, total=False):
    customer_id: str
    issue_type: str
    user_query: str

    customer_profile: Dict[str, Any]
    plan_steps: List[str]

    kpi_result: Dict[str, Any]
    incident_result: Dict[str, Any]
    billing_result: Dict[str, Any]
    rag_results: List[Dict[str, Any]]

    diagnosis: str
    next_action: str
    escalation_required: bool

    final_response: str

    prompt_tokens: int
    completion_tokens: int
    estimated_cost_usd: float

    trace_id: str

## Step 11 — Planner node

In [9]:
TRACE_CONTEXT = {"current": None}

def planner_node(state: CapstoneState) -> Dict[str, Any]:
    trace = TRACE_CONTEXT["current"]
    customer = CUSTOMERS[state["customer_id"]]

    plan_steps = [
        "Load customer profile",
        "Check KPI API",
        "Check Incident API",
        "Check Billing API",
        "Run RAG over SOP guidance",
        "Make decision",
        "Generate response",
    ]

    output = {
        "customer_profile": customer,
        "plan_steps": plan_steps,
    }

    add_run(trace, "planner", {"customer_id": state["customer_id"], "issue_type": state["issue_type"]}, output, prompt_tokens=20, completion_tokens=15)
    return output

## Step 12 — Tool nodes

In [10]:
def kpi_node(state: CapstoneState) -> Dict[str, Any]:
    trace = TRACE_CONTEXT["current"]
    city = state["customer_profile"]["city"]
    result = kpi_api(city, state["issue_type"])
    add_run(trace, "kpi_tool", {"city": city, "issue_type": state["issue_type"]}, result)
    return {"kpi_result": result}

def incident_node(state: CapstoneState) -> Dict[str, Any]:
    trace = TRACE_CONTEXT["current"]
    city = state["customer_profile"]["city"]
    result = incident_api(city, state["issue_type"])
    add_run(trace, "incident_tool", {"city": city, "issue_type": state["issue_type"]}, result)
    return {"incident_result": result}

def billing_node(state: CapstoneState) -> Dict[str, Any]:
    trace = TRACE_CONTEXT["current"]
    result = billing_api(state["customer_id"])
    add_run(trace, "billing_tool", {"customer_id": state["customer_id"]}, result)
    return {"billing_result": result}

## Step 13 — RAG node

In [11]:
def rag_node(state: CapstoneState) -> Dict[str, Any]:
    trace = TRACE_CONTEXT["current"]
    city = state["customer_profile"]["city"]
    result = mock_rag_search(state["user_query"], city_filter=city, issue_filter=state["issue_type"], top_k=2)
    add_run(trace, "rag_retrieval", {"query": state["user_query"], "city_filter": city, "issue_filter": state["issue_type"]}, {"results": result})
    return {"rag_results": result}

## Step 14 — Decision node

In [12]:
def decision_node(state: CapstoneState) -> Dict[str, Any]:
    trace = TRACE_CONTEXT["current"]

    kpi = state["kpi_result"]
    incident = state["incident_result"]
    billing = state["billing_result"]
    rag = state["rag_results"]

    if incident["incident_active"]:
        diagnosis = f"Confirmed incident: {incident['incident_summary']}"
        next_action = "Communicate active incident and escalate to network operations."
        escalation_required = True
    elif kpi["tower_utilization_pct"] >= 90 or kpi["packet_loss_pct"] >= 5:
        diagnosis = rag[0]["content"] if rag else "Likely network degradation based on KPI signals."
        next_action = "Follow SOP guidance and notify NOC if repeated complaints continue."
        escalation_required = True
    elif billing["billing_anomaly"]:
        diagnosis = f"Billing concern detected: {billing['billing_summary']}"
        next_action = "Raise a billing support case and explain overage/usage details."
        escalation_required = False
    else:
        diagnosis = "No major network or billing issue detected from current signals."
        next_action = "Provide self-help guidance and monitor."
        escalation_required = False

    prompt_tokens = estimate_tokens(diagnosis + " " + next_action)
    completion_tokens = 40
    cost = round((prompt_tokens / 1000) * 0.0005 + (completion_tokens / 1000) * 0.0015, 6)

    output = {
        "diagnosis": diagnosis,
        "next_action": next_action,
        "escalation_required": escalation_required,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "estimated_cost_usd": cost,
    }

    add_run(trace, "decision", {"kpi": kpi, "incident": incident, "billing": billing, "rag_top": rag[:1]}, output, prompt_tokens=prompt_tokens, completion_tokens=completion_tokens)
    return output

## Step 15 — Response node

In [13]:
def response_node(state: CapstoneState) -> Dict[str, Any]:
    trace = TRACE_CONTEXT["current"]

    profile = state["customer_profile"]
    final_response = (
        f"Customer: {profile['name']}\n"
        f"City: {profile['city']}\n"
        f"Plan: {profile['plan']}\n"
        f"Issue Type: {state['issue_type']}\n"
        f"Diagnosis: {state['diagnosis']}\n"
        f"Next Action: {state['next_action']}\n"
        f"Escalation Required: {state['escalation_required']}\n"
        f"Prompt Tokens: {state['prompt_tokens']}\n"
        f"Completion Tokens: {state['completion_tokens']}\n"
        f"Estimated Cost USD: {state['estimated_cost_usd']}"
    )

    add_run(trace, "response", {"diagnosis": state["diagnosis"], "next_action": state["next_action"]}, {"final_response": final_response})
    return {"final_response": final_response}

## Step 16 — Build the LangGraph workflow

In [14]:
builder = StateGraph(CapstoneState)

builder.add_node("planner", planner_node)
builder.add_node("kpi", kpi_node)
builder.add_node("incident", incident_node)
builder.add_node("billing", billing_node)
builder.add_node("rag", rag_node)
builder.add_node("decision", decision_node)
builder.add_node("response", response_node)

builder.set_entry_point("planner")
builder.add_edge("planner", "kpi")
builder.add_edge("kpi", "incident")
builder.add_edge("incident", "billing")
builder.add_edge("billing", "rag")
builder.add_edge("rag", "decision")
builder.add_edge("decision", "response")
builder.add_edge("response", END)

capstone_graph = builder.compile()
print("IBM Telco Intelligent Agent workflow compiled")

IBM Telco Intelligent Agent workflow compiled


## Step 17 — Run the capstone workflow

In [15]:
input_state = {
    "customer_id": "CUST_1001",
    "issue_type": "slow_internet",
    "user_query": "My mobile internet is slow in Pune and my bill is high. What should support do next?",
}

trace = start_trace("ibm_telco_intelligent_agent", input_state)
TRACE_CONTEXT["current"] = trace

result = capstone_graph.invoke(input_state)
print(result["final_response"])

Customer: Ravi Patil
City: Pune
Plan: Premium 5G
Issue Type: slow_internet
Diagnosis: If unusual bill is reported, verify usage spike, overage, active plan, and raise billing support case if needed.
Next Action: Follow SOP guidance and notify NOC if repeated complaints continue.
Escalation Required: True
Prompt Tokens: 28
Completion Tokens: 40
Estimated Cost USD: 7.4e-05


## Step 18 — Inspect trace

In [16]:
pprint(trace)

{'initial_input': {'customer_id': 'CUST_1001',
                   'issue_type': 'slow_internet',
                   'user_query': 'My mobile internet is slow in Pune and my '
                                 'bill is high. What should support do next?'},
 'runs': [{'completion_tokens': 15,
           'input': {'customer_id': 'CUST_1001', 'issue_type': 'slow_internet'},
           'node_name': 'planner',
           'output': {'customer_profile': {'city': 'Pune',
                                           'customer_id': 'CUST_1001',
                                           'name': 'Ravi Patil',
                                           'plan': 'Premium 5G',
                                           'recent_complaints': 2},
                      'plan_steps': ['Load customer profile',
                                     'Check KPI API',
                                     'Check Incident API',
                                     'Check Billing API',
                                

## Step 19 — Capstone evaluation

In [17]:
def evaluate_accuracy(result_state: Dict[str, Any], expected_keywords: List[str]) -> float:
    text = result_state["final_response"].lower()
    matches = sum(1 for kw in expected_keywords if kw.lower() in text)
    return matches / max(len(expected_keywords), 1)

def evaluate_cost(trace: Dict[str, Any]) -> float:
    return trace["total_cost_usd"]

def evaluate_performance(trace: Dict[str, Any]) -> int:
    total_latency = 0
    for run in trace["runs"]:
        total_latency += estimate_latency_ms(run["node_name"])
    return total_latency

In [18]:
expected_keywords = ["congestion", "notify noc", "billing"]
accuracy_score = evaluate_accuracy(result, expected_keywords)
cost_score = evaluate_cost(trace)
latency_score = evaluate_performance(trace)

evaluation_summary = {
    "accuracy_score": accuracy_score,
    "estimated_total_cost_usd": cost_score,
    "estimated_total_latency_ms": latency_score,
}

pprint(evaluation_summary)

{'accuracy_score': 0.6666666666666666,
 'estimated_total_cost_usd': 0.000106,
 'estimated_total_latency_ms': 245}


## Step 20 — Tiny evaluation dataset

In [19]:
evaluation_cases = [
    {
        "case_id": "CASE001",
        "customer_id": "CUST_1001",
        "issue_type": "slow_internet",
        "user_query": "My mobile internet is slow in Pune and my bill is high.",
        "expected_keywords": ["congestion", "billing", "notify noc"],
    },
    {
        "case_id": "CASE002",
        "customer_id": "CUST_2001",
        "issue_type": "packet_loss",
        "user_query": "There is packet loss affecting browsing in Mumbai.",
        "expected_keywords": ["incident", "network operations", "packet loss"],
    },
]

pd.DataFrame(evaluation_cases)

,case_id,customer_id,issue_type,user_query,expected_keywords
0,CASE001,CUST_1001,slow_internet,My mobile internet is slow in Pune and my bill...,"[congestion, billing, notify noc]"
1,CASE002,CUST_2001,packet_loss,There is packet loss affecting browsing in Mum...,"[incident, network operations, packet loss]"


## Step 21 — Evaluate all capstone cases

In [20]:
rows = []

for case in evaluation_cases:
    trace = start_trace(case["case_id"], case)
    TRACE_CONTEXT["current"] = trace

    state = {
        "customer_id": case["customer_id"],
        "issue_type": case["issue_type"],
        "user_query": case["user_query"],
    }

    result = capstone_graph.invoke(state)

    rows.append({
        "case_id": case["case_id"],
        "accuracy": evaluate_accuracy(result, case["expected_keywords"]),
        "estimated_cost_usd": evaluate_cost(trace),
        "estimated_latency_ms": evaluate_performance(trace),
    })

eval_df = pd.DataFrame(rows)
eval_df

,case_id,accuracy,estimated_cost_usd,estimated_latency_ms
0,CASE001,0.666667,0.000106,245
1,CASE002,1.000000,0.000100,245


## Step 22 — Capstone summary

What this capstone demonstrated:

### Design
- use case definition
- enterprise workflow thinking
- MCP-style tool/resource/prompt structure

### Build
- LangGraph workflow
- telecom tools
- Milvus-style RAG retrieval
- decision node
- response / escalation logic

### Observability
- LangSmith-style tracing concepts

### Evaluation
- accuracy
- cost
- performance

## Step 23 — How to upgrade this capstone in a real project

1. Replace mock RAG with real Milvus / Zilliz Cloud
2. Add real LangSmith tracing
3. Add real tool contracts with validation
4. Add ticketing and CRM integration
5. Add HITL approval for high-risk escalations
6. Add faithfulness and hallucination evaluation
7. Split into planner / executor / validator subgraphs

## Step 24 — Exercises

1. Add a real ticket creation node
2. Add a validator node after response
3. Add caching for repeated queries
4. Add a billing-only route
5. Add LangSmith environment variables for real tracing

# Final takeaway

This capstone project shows how a telecom intelligent agent can be built as a structured workflow:

```text
User Query
   ↓
Planner
   ↓
Tools
   - KPI API
   - Incident API
   - Billing API
   ↓
RAG (SOP)
   ↓
Decision Node
   ↓
Response / Escalation
```

That is much closer to a real enterprise AI system than a simple chatbot.